In [1]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine, event
import urllib
import os

# --- CONFIGURATION ---
# 1. Paths and Connection Details
SQLITE_DB_PATH = r"D:\FinTech\Data\Processed\FinTech_Warehouse.db"
SERVER_NAME = r'DESKTOP-BMSUUJ7\SQLEXPRESS' # Added 'r' to handle the backslash
DATABASE_NAME = 'FinTech_DWH'
DRIVER = 'ODBC Driver 17 for SQL Server' 

# --- STEP 1: LOAD FROM SQLITE (SILVER LAYER) ---
print(f"Reading data from SQLite: {SQLITE_DB_PATH}...")

if not os.path.exists(SQLITE_DB_PATH):
    raise FileNotFoundError(f"The SQLite file was not found at {SQLITE_DB_PATH}. Please check the path.")

try:
    sqlite_conn = sqlite3.connect(SQLITE_DB_PATH)
    df_cust = pd.read_sql("SELECT * FROM silver_customers", sqlite_conn)
    df_tx = pd.read_sql("SELECT * FROM silver_transactions", sqlite_conn)
    sqlite_conn.close()
    print(f"✅ Loaded {len(df_cust)} customers and {len(df_tx)} transactions.")
except Exception as e:
    print(f"❌ Error reading SQLite: {e}")
    raise

# --- STEP 2: SETUP SQL SERVER ENGINE ---
# Note: Added TrustServerCertificate for compatibility with newer SQL setups
connection_string = (
    f'DRIVER={{{DRIVER}}};'
    f'SERVER={SERVER_NAME};'
    f'DATABASE={DATABASE_NAME};'
    f'Trusted_Connection=yes;'
    f'TrustServerCertificate=yes;' 
)

params = urllib.parse.quote_plus(connection_string)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# PERFORMANCE HACK: Enables fast batch inserts
@event.listens_for(engine, "before_cursor_execute")
def receive_before_cursor_execute(conn, cursor, statement, parameters, context, executemany):
    if executemany:
        cursor.fast_executemany = True

# --- STEP 3: MIGRATE DATA TO SQL SERVER ---
print(f"🚀 Starting migration to SQL Server: {DATABASE_NAME}...")

try:
    # Migrate Customers
    df_cust.to_sql(
        name='silver_customers', 
        con=engine, 
        if_exists='replace', 
        index=False, 
        chunksize=10000
    )
    print("✅ 'silver_customers' table uploaded successfully.")

    # Migrate Transactions
    df_tx.to_sql(
        name='silver_transactions', 
        con=engine, 
        if_exists='replace', 
        index=False, 
        chunksize=10000
    )
    print("✅ 'silver_transactions' table uploaded successfully.")

    print("\n" + "="*40)
    print("MIGRATION COMPLETE: Silver Layer is now in SQL Server")
    print("="*40)

except Exception as e:
    print(f"❌ Migration failed: {e}")
    print("\nTroubleshooting Tips:")
    print(f"1. Make sure you created the database '{DATABASE_NAME}' in SSMS.")
    print(f"2. Check if '{DRIVER}' is the version you have (check 'ODBC Data Sources' in Windows).")

Reading data from SQLite: D:\FinTech\Data\Processed\FinTech_Warehouse.db...
✅ Loaded 94728 customers and 1000000 transactions.
🚀 Starting migration to SQL Server: FinTech_DWH...
✅ 'silver_customers' table uploaded successfully.
✅ 'silver_transactions' table uploaded successfully.

MIGRATION COMPLETE: Silver Layer is now in SQL Server
